<a href="https://colab.research.google.com/github/bugristaya/comp_ling/blob/main/%D0%91%D1%8B%D0%BA%D0%BE%D0%B2%D0%B0_%D0%9A%D1%83%D0%B7%D0%BD%D0%B5%D1%86%D0%BE%D0%B2%D0%B0_%D0%A1%D0%BE%D0%B7%D0%B4%D0%B0%D0%BD%D0%B8%D0%B5_%D0%B2%D0%B5%D0%BA%D1%82%D0%BE%D1%80%D0%BD%D0%BE%D0%B9_%D0%B1%D0%B0%D0%B7%D1%8B_%D0%B4%D0%B0%D0%BD%D0%BD%D1%8B%D1%85.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Тема:** работа с векторной БД ChromaDB и кастомным RAG-пайплайном


**Задачи:**

Разработка функционала RAG с ChromaDB
1. Загрузить набор документов (например, датасеты для RAG из HuggingFace) и создать текстовые чанки
1. С помощью sentence-transformers сгенерировать эмбеддинги для каждого чанка
1. Сохранить эмбеддинги и метаданные в локальную базу данных ChromaDB
1. Реализовать функцию семантического поиска, которая ищет топ-N похожих чанков
1. Сформировать промпт и подать его на вход LLM для генерации итогового ответа

**Библиотеки:** chromadb, sentence-transformers, API для запросов к LLM

**Ожидаемый результат:** Colab-ноутбук, показывающий полный цикл разработки RAG-пайплайна от создания коллекции в ChromaDB до генерации ответа с LLM

## Набор документов

выберем подходящее на HF

In [ ]:
# используем токен HF для аутентификации
!hf auth login

A new version of huggingface_hub (1.7.2) is available! You are using version 1.7.1.
To update, run: pip install -U huggingface_hub


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? [y/N]: y
Token is valid (permission: write).
The token `comp_ling` has been saved to /root/.cache/huggingface/sto

In [ ]:
# устанавливаем все необходимое
!pip install langchain langchain_text_splitters
!pip install llama-index-core llama-index-embeddings-huggingface llama-index-llms-openrouter
!pip install langchain-openai langgraph
!pip install datasets pandas
!pip install llama-index-llms-langchain
!pip install langchain-core langchain-community
!pip install chromadb sentence-transformers
!pip install langchain-huggingface

  Using cached chromadb-1.5.5-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.2 kB)
  Using cached build-1.4.0-py3-none-any.whl.metadata (5.8 kB)
  Using cached pybase64-1.4.3-cp312-cp312-manylinux1_x86_64.manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_5_x86_64.whl.metadata (8.7 kB)
  Using cached onnxruntime-1.24.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (5.2 kB)
  Using cached opentelemetry_exporter_otlp_proto_grpc-1.40.0-py3-none-any.whl.metadata (2.6 kB)
  Using cached pypika-0.51.1-py2.py3-none-any.whl.metadata (51 kB)
  Using cached bcrypt-5.0.0-cp39-abi3-manylinux_2_34_x86_64.whl.metadata (10 kB)
  Using cached kubernetes-35.0.0-py2.py3-none-any.whl.metadata (1.7 kB)
  Using cached pyproject_hooks-1.2.0-py3-none-any.whl.metadata (1.3 kB)
  Using cached durationpy-0.10-py3-none-any.whl.metadata (340 bytes)
  Using cached opentelemetry_exporter_otlp_proto_common-1.40.0-py3-none-any.whl.metadata (1.9 kB)
  Using cached opent

In [ ]:
# импортируем все необходимое
from datasets import load_dataset
import pandas as pd

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from google.colab import userdata


In [ ]:
# загружаем датасет
ds = load_dataset("Amod/mental_health_counseling_conversations")
print("Датасет загружен. Размер:", len(ds['train']))

README.md: 0.00B [00:00, ?B/s]

combined_dataset.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/3512 [00:00<?, ? examples/s]

Датасет загружен. Размер: 3512


In [ ]:
# выводим первые наблюдения из датесета
df = pd.DataFrame({
    'context': ds['train']['Context'],
    'response': ds['train']['Response']
})

df.head(3)

,context,response
0,I'm going through some things with my feelings...,"If everyone thinks you're worthless, then mayb..."
1,I'm going through some things with my feelings...,"Hello, and thank you for your question and see..."
2,I'm going through some things with my feelings...,First thing I'd suggest is getting the sleep y...


In [ ]:
# вытаскиваем текст из датасета и преобразуем в Document-объекты
documents = []
for idx, row in df.iterrows():
    full_text = f"Контекст\Ответ: {row['context']} {row['response']}"  # объединяем текст: контекст + ответ психотерапевта
    doc = Document(
        page_content=full_text,
        metadata={
            "id": idx,
            "source": "mental_health_counseling",
            "context": row['context'],
            "response": row['response']
        }
    )
    documents.append(doc)

print(f"Создано {len(documents)} Document-объектов.")

Создано 3512 Document-объектов.


## Разделение на чанки

In [ ]:
text_splitter = RecursiveCharacterTextSplitter( # этот тип splitter'a был выбран, потому что он позволяет разбивать текст по знакам препинания, а не просто по количеству символов
    chunk_size=150, # в датасете встречаются как короткие, так и длинные предложения. Такой размер чанка вмещает 1–2 предложения
    chunk_overlap=30, # 20% от размера чанка выбрано, чтобы не потерять контекст
    separators=['.', '!', '?', '\n', ' ']
)

chunks = text_splitter.split_documents(documents) # сохраняем в chunks

In [ ]:
chunks = text_splitter.split_documents(documents)
print(f"Создано чанков: {len(chunks)}")
print(f"Пример чанка:{chunks[33].page_content[:200]}...") # выводим случайный чанк для примера

Создано чанков: 46167
Пример чанка:. As for suicidal thoughts, I am very glad to read that this has not happened to you...


## Эмбеддинги: загружаем модель

In [ ]:
# выбираем модель
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2', device='cpu')  # загружаем модель
texts = [chunk.page_content for chunk in chunks]
embeddings = model.encode(texts, batch_size=128, show_progress_bar=True) # создаем батчи для оптимизации

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/361 [00:00<?, ?it/s]

## Сохранение эмбеддингов и метаданных в БД

In [ ]:
persist_directory = "./chromaBD"
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory=persist_directory
)
print(f"База данных сохранена в {persist_directory}")

База данных сохранена в ./chromaBD


## Функция семантического поиска

Score в ChromaDB — расстояние, а не близость

0.0 = идеальное совпадение

чем меньше score, тем более похож результат на запрос

In [ ]:
def semantic_search(query: str, top_k: int = 5):
    results = vectorstore.similarity_search_with_score(query, k=top_k) # similarity_search_with_score был выбран, потому что он показывает, насколько точно найденный фрагмент соотвествует к запросу
    return results

In [ ]:
# тестирование функции поиска
test_results = semantic_search("eating disorder", 2)
print("Тест поиск по запросу 'eating disorder':")
for doc, score in test_results:
    print(f"Score: {score:.4f} -> {doc.page_content[:100]}...")

Тест поиск по запросу 'eating disorder':
Score: 0.4829 -> ? Eating disorders usually result from a sense of insecurity about who the person is, whether they a...
Score: 0.5323 -> harmful emotions sometimes associated with disordered eating...


## Промпт и генерация итогового ответа

In [ ]:
# подключаем LLM через OpenRouter
OPEN_ROUTER_API_KEY = userdata.get('OPEN_ROUTER_KEY')

llm = ChatOpenAI(
    api_key=OPEN_ROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1",
    model="google/gemma-3-4b-it:free",
)

In [ ]:
# создаем функцию для генерации ответов
def answer_question(query: str, top_k: int = 5):
    # ищем релевантные чанки
    results = vectorstore.similarity_search_with_score(query, k=top_k)
    context_text = "\n\n---\n\n".join([doc.page_content for doc, _ in results])

    # формируем промпт
    prompt = f""" Based on one-on-one mental health counseling conversations, answer the question accurately.

    Context: {context_text} Question: {query} Answer:"""

    # вызываем LLM и получаем ответ
    answer = llm.invoke(prompt).content

    # 4. Возврат результата с метаданными
    return {
        "answer": answer,
        "used_chunks": [(doc.page_content, score) for doc, score in results],
        "scores": [score for _, score in results]
    }

In [ ]:
questions = [
    "What technique is recommended for dealing with nightmares?",
    "What general strategies are recommended for coping with anxiety and stress?",
    "What exactly does 'normalizing eating' involve, and why is it important?"
]

for q in questions:
    print(f"Вопрос: {q} ")
    result = answer_question(q, top_k=3)
    print(f"Ответ:\n{result['answer']}")
    print("Использованные чанки (расстояние, меньше = лучше):")
    for i, (chunk, score) in enumerate(result['used_chunks']):
        print(f"{i+1}. score={score:.4f} -> {chunk}\n")

Вопрос: What technique is recommended for dealing with nightmares? 
Ответ:
The text recommends trying “Nightmare Rescripting” or “Nightmare Exposure,” which can be found through a Google search. It also suggests deep breathing as a helpful technique.
Использованные чанки (расстояние, меньше = лучше):
1. score=0.3646 -> . You may also want to try deep breathing.4. There are self-help ideas for managing bad nightmares

2. score=0.3646 -> . You may also want to try deep breathing.4. There are self-help ideas for managing bad nightmares

3. score=0.5332 -> . If you Google search "Nightmare Rescripting" or "Nightmare Exposure" you may find some ideas and instructions on how to manage dreams

Вопрос: What general strategies are recommended for coping with anxiety and stress? 
Ответ:
Okay, based on the context provided – that we’re discussing ways to deal with anxiety – here’s a response to your question about general strategies for coping with anxiety and stress, reflecting what’s often disc

Вопрос 1: What technique is recommended for dealing with nightmares?

Для ответа были использованы чанки, содержащие ключевые слова «technique» и «nightmares». Благодаря семантическому поиску были найдены релевантные фрагменты, которые описывают дыхательные упражнения и технику «перезаписи кошмаров». Однако важно отметить, что чанки 1 и 2 повторяются, но в данном примере это не привело к ошибочному ответу.

Вопрос 2: What general strategies are recommended for coping with anxiety and stress?

При ответе о стратегиях борьбы с тревогой и стрессом были возвращены фрагменты, которые касаются темы вопроса, но ни один из них не содержит детального списка стратегий. Вероятно, развёрнутый ответ LLM основан скорее на общих знаниях модели.

Вопрос 3: What exactly does 'normalizing eating' involve, and why is it important?

Вопрос содержит термин «normalizing eating», поэтому при ответе были использованы чанки, содержащие полностью или частично это выражение. Этот ответ является наиболее корректным благодаря определению и объяснению важности этого подхода для преодоления расстройств пищевого поведения. Однако контекст в чанках был достаточно кратким.

Таким образом, основной проблемой системы является дублирование чанков, несмотря на то, что база Chroma очищалась перед запуском кода. Стоит отметить, что при семантическом поиске были найдены релевантные чанки, а LLM генерирует связные ответы. Также система обозначает невозможность дать развёрнутый ответ при большом score, что свидетельствует о корректной работе механизма оценки релевантности. В дальнейшем улучшить работу системы можно с помощью увеличения размеров чанков и работы с базой chroma для избежания дублирования чанков и формирования качественного ответа.

# Критерии оценивания

Работа проверяется по следующим критериям (максимум 10 баллов):

### Загрузка и подготовка данных (2 балла)
- [ ] 0.5 балла: выбран датасет (arXiv, IMDB, AG News или другой)
- [ ] 0.5 балла: загружено не менее 100 записей/статей
- [ ] 0.5 балла: тексты успешно извлечены из источника
- [ ] 0.5 балла: данные конвертированы в Document-объекты с соответствующими метаданными

### Чанкинг (2 балла)
- [ ] 0.5 балла: выбран подходящий тип сплиттера (RecursiveCharacterTextSplitter, HTMLHeaderTextSplitter и т.д.)
- [ ] 0.5 балла: обоснован выбор размера чанка и перекрытия (например, "512 токенов, overlap 20% для сохранения контекста")
- [ ] 0.5 балла: чанки созданы и не содержат явных артефактов (оборванных слов)
- [ ] 0.5 балла: количество чанков соответствует ожидаемому (не 1 и не 100500 на документ)

### Векторное хранилище (1 балл)
- [ ] 0.5 балла: выбрана адекватная embedding_model
- [ ] 0.5 балла: эмбеддинги и метаданные успешно сохранены в БД Chroma

### Функция семантического поиска (1,5 балла)
- [ ] 0.5 балла: обоснован выбор similarity_search / similarity_search_with_score
- [ ] 0.5 балла: успешно реализована semantic_search()
- [ ] 0.5 балла: протестирован вызов функции

### Промпт и генерация итогового ответа (1,5 балла)
- [ ] 0.5 балла: подключена LLM
- [ ] 0.5 балла: корректно составлены query, context, prompt
- [ ] 0.5 балла: ответ генерируется на основе найденных чанков (видно по содержанию)

### Тестирование и анализ (2 балла)
- [ ] 0.5 балла: задано минимум 3 разнотипных вопроса (фактический, обобщающий, уточняющий)
- [ ] 0.5 балла: для каждого вопроса показан и проанализирован ответ
- [ ] 0.5 балла: в анализе указано, какие чанки использовались и почему
- [ ] 0.5 балла: сделан вывод о качестве работы системы (что получилось, что нет, гипотезы почему)
